# D12 - il blackout totale geografico, otto mondi

**Autosufficiente.** Carica solo questo file e premi **Esegui tutto**. Quando compare il popup di Google Drive, autorizza l'accesso: e' l'unico momento in cui serve una persona.

**Runtime:** GPU T4. **Durata:** 8 fit, circa un'ora e mezza.

**Cosa fa.** Rigenera gli otto mondi dello studio di copertura (semi 42, 101-107) con il blackout totale - tutti e sette i canali a spesa esattamente zero in cinque regioni fisse (Lazio, Emilia-Romagna, Calabria, Abruzzo, Basilicata: una per strato di popolazione, 23,39% della popolazione) per due blocchi di otto settimane (9-16 e 53-60) - e li fitta con lo stesso runner e la stessa configurazione di tutta la griglia (96 nodi, keep 2000, 4 catene, seme MCMC 42, revenue_per_kpi 40, prior LogNormal(0,2; 0,9)). Generatore = main del repo pubblico + patch A + G1 + G2 + G3 + G4 + **G6** (`--blackout`), impronta `8f86eb032d0113c0`; runner = R7 + **R8** (disegno `blackout`, `--controllo-baseline`).

**Che cosa lo distingue da tutti i disegni precedenti.** La spesa tolta **non viene ridistribuita**: la spesa nazionale settimanale scende (del 23,65% nelle sedici settimane trattate sul mondo 42, del 3,2-3,7% sul biennio). Nessuno degli undici disegni gia' eseguiti cambia *quanta* pubblicita' c'e': cambiano solo dove e quando. D12 e' l'unico che muove il livello.

**Il punto tecnico decisivo.** Al passo 7 il generatore ricalibra `beta` per centrare la quota di contributo dichiarata: con meno spesa il contributo vero resterebbe **identico per costruzione** e il disegno misurerebbe zero. La patch G6 tiene `beta` e la mezza saturazione `ec` fissi a quelli del mondo base dello stesso seme. Verificato prima dei fit: senza G6 il contributo vero non si muove di una cifra; con G6 scende del 3,4%. Conseguenza da dichiarare accanto a ogni cifra: la verita' di questi mondi e' la superficie di risposta implicita nei parametri veri del mondo base.

**Attenzione alla metrica.** La quota vera dei media in questi mondi **non e' 18,5%**: vale fra 17,94% e 18,04%. `E` si calcola contro la quota vera di questo mondo, che il runner scrive nella colonna `quota_media_vera_pct`.

**Il costo.** Il blackout costa davvero: fra 114.433 e 140.428 EUR di spesa non fatta e fra 4.841 e 5.842 candidature vere non arrivate, a seconda del mondo. Il runner lo scrive riga per riga nelle colonne `candidature_vere_perse` e `quota_media_vera_base_pct`.

**Scrive in `MMM_griglia_D12`**, cartella nuova: `MMM_griglia`, `MMM_griglia_D10` e le altre non vengono toccate.

**Alla fine:** scarica la cartella per intera e riporta i riepiloghi in chat. Il ramo si legge solo sulla pre-registrazione `PREREGISTRAZIONE_D12_blackout_totale.md`, dopo il commit dei risultati.


In [ ]:
# ============================================================================
# CELLA 1 - PRE-VOLO (otto controlli) E SETUP: repo, patch, runner, attese
# ============================================================================
# Questa cella non fitta niente: verifica che una notte di run abbia senso
# PRIMA di cominciarla, e prepara il codice. Se anche uno solo degli otto
# controlli fallisce si ferma qui, scrive cosa fare e NON definisce la
# variabile che sblocca la cella del ciclo.
import base64, csv, hashlib, io, json, os, shutil, subprocess, sys, tempfile, time

A = chr(39)          # apostrofo
SPUNTA = chr(10003)
CROCE = chr(10007)

# ============================================================================
# COSTANTI - tutto quello che si puo' voler cambiare sta qui e solo qui
# ============================================================================
REPO = '/content/mmm-bayesiano-recruiting'
URL_REPO = 'https://github.com/Giacomod2001/mmm-bayesiano-recruiting'
CARTELLA_DRIVE = '/content/drive/MyDrive/MMM_griglia_D12'      # NUOVA: le cartelle dei run precedenti non si toccano
CARTELLA_COPERTURA = '/content/drive/MyDrive/MMM_copertura'    # solo LETTA: i semi dello studio osservativo
# D12: il BLACKOUT TOTALE geografico, otto mondi. Unico disegno di questo
# notebook. Tutti e sette i canali a spesa esattamente zero nelle stesse
# cinque regioni (una per strato di popolazione: Lazio, Emilia-Romagna,
# Calabria, Abruzzo, Basilicata) per due blocchi di otto settimane (9-16 e
# 53-60), e la spesa tolta NON viene ridistribuita. E' l'unico disegno della
# griglia che muove il LIVELLO dei media e non solo la loro distribuzione.
# beta e la mezza saturazione ec restano quelli del mondo base dello stesso
# seme (patch G6): senza, la ricalibrazione del passo 7 del generatore
# riporterebbe il contributo vero al suo valore di sempre e il disegno
# misurerebbe zero. La regola di lettura sta nel paragrafo 4 della
# pre-registrazione, fissata prima. La griglia e i notebook D10 e D11 non si
# toccano: questo scrive altrove.
DISEGNI_ATTIVI = ['D12']
MONDI_ATTIVI = [42, 101, 102, 103, 104, 105, 106, 107]           # mai altri
KEEP = 2000
VERSIONE_MERIDIAN = '1.8.0'
# Solo per il collaudo locale (Meridian finto): sottoinsiemi via ambiente.
if os.environ.get('GRIGLIA_DISEGNI'):
    DISEGNI_ATTIVI = [d.strip() for d in os.environ['GRIGLIA_DISEGNI'].split(',') if d.strip()]
if os.environ.get('GRIGLIA_MONDI'):
    MONDI_ATTIVI = [int(m) for m in os.environ['GRIGLIA_MONDI'].split(',') if m.strip()]

DIR_PREREG = os.path.join(CARTELLA_DRIVE, 'preregistrazioni')
ATTESI_DRIVE = os.path.join(CARTELLA_DRIVE, 'attesi_griglia.csv')
IMPRONTE_DRIVE = os.path.join(CARTELLA_DRIVE, 'impronte_griglia.csv')
RIEPILOGO = os.path.join(CARTELLA_DRIVE, 'griglia_riepilogo.csv')
CONTEGGI = os.path.join(CARTELLA_DRIVE, 'griglia_conteggi.csv')
GEN = os.path.join(REPO, 'genera_dati_simulati.py')
RUNNER = os.path.join(REPO, 'studio_copertura_runner.py')

# Il materiale incorporato: le sostituzioni A del generatore (--seed-mondo, 4
# decimali, --sanita), le patch G1/G2/G3/G4/G6 come diff unificati, l'impronta
# sha256 del generatore che ne risulta, il runner R8 completo, la tabella dei
# run con le attese per riga, il file degli attesi, le pre-registrazioni e le
# impronte dei file media dichiarate nella pre-registrazione di D12.
_BLOB = (
    '...',
    'ZCJ9fX0=',
)
_dati = json.loads(base64.b64decode(''.join(_BLOB)).decode('utf-8'))
TABELLA = _dati['tabella']
RUN_PER_NOME = {r['nome']: r for r in TABELLA}
DISEGNI = _dati['disegni']                    # [[codice, titolo, csv, prereg, attivo_default], ...]
INFO_DISEGNO = {d[0]: {'titolo': d[1], 'csv': d[2], 'prereg': d[3]} for d in DISEGNI}
CONFIG = _dati['config']
ATTESI_CSV_TESTO = _dati['attesi_csv']
VERSIONE_ATTESI = int(_dati['versione_attesi'])
PREREGISTRAZIONI = _dati['preregistrazioni']  # {nome file: testo}
OSSERVATIVO = {int(k): v for k, v in _dati['osservativo'].items()}   # {seme: riga dello studio osservativo}
_sconosciuti = [d for d in DISEGNI_ATTIVI if d not in INFO_DISEGNO]
if _sconosciuti:
    raise SystemExit('DISEGNI_ATTIVI contiene codici sconosciuti: ' + str(_sconosciuti) +
                     '. Ammessi: ' + ', '.join(INFO_DISEGNO))
_mondi_noti = sorted({int(r['seed_mondo']) for r in TABELLA})
_fuori = [m for m in MONDI_ATTIVI if m not in _mondi_noti]
if _fuori:
    raise SystemExit('MONDI_ATTIVI contiene semi non pre-registrati: ' + str(_fuori) +
                     '. I mondi sono ' + str(_mondi_noti) + ' e mai altri.')


def csv_di(run):
    return os.path.join(CARTELLA_DRIVE, INFO_DISEGNO[run['codice']]['csv'])


def affiancato(percorso_csv, suffisso):
    return percorso_csv[:-4] + suffisso


# ============================================================================
# IL CONTROLLO DI UNA RIGA - usato dal ciclo (CELLA 2) e dal pre-volo
# ============================================================================

def leggi_csv(percorso):
    if not (os.path.exists(percorso) and os.path.getsize(percorso) > 0):
        return []
    with open(percorso, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def _numero(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def leggi_impronte():
    """{nome_run: impronta} registrate al primo fit: al rilancio diventano l'attesa."""
    return {r['nome']: r['impronta'] for r in leggi_csv(IMPRONTE_DRIVE) if r.get('impronta')}


def controllo_config(run, riga, impronte):
    """La riga del runner contro la SUA riga della tabella: configurazione del
    fit E del disegno. Una differenza = anomalia."""
    p = []
    if riga.get('esito') != 'ok':
        p.append('esito=' + str(riga.get('esito')) + ' (' + str(riga.get('errore'))[:160] + ')')
        return p
    attese = [('n_knots', CONFIG['n_knots']), ('keep', CONFIG['keep']),
              ('n_chains', CONFIG['n_chains']), ('adapt', CONFIG['adapt']),
              ('burnin', CONFIG['burnin']), ('seed_mcmc', CONFIG['seed_mcmc']),
              ('revenue_per_kpi', CONFIG['revenue_per_kpi']),
              ('prior_sigma', run['prior_sigma']),
              ('decimali_stagionalita', CONFIG['decimali']),
              ('seed_mondo', run['seed_mondo'])]
    if run['prior_mu'] not in ('', None):
        attese.append(('prior_mu', run['prior_mu']))
    for campo, atteso in attese:
        v = _numero(riga.get(campo))
        if v is None or abs(v - float(atteso)) > 1e-9:
            p.append(campo + '=' + str(riga.get(campo)) + ' invece di ' + str(atteso))
    if str(riga.get('prior_tipo')) != str(run['prior_tipo']):
        p.append('prior_tipo=' + str(riga.get('prior_tipo')) + ' invece di ' + str(run['prior_tipo']))
    if run['prior_tipo'] == 'ancorato-benchmark' and str(riga.get('prior_mu_per_canale', '')).count('=') != 7:
        p.append('prior_mu_per_canale non ha 7 canali: ' + str(riga.get('prior_mu_per_canale'))[:80])
    for campo in ('disegno', 'parametri_disegno', 'canali_trattati'):
        if str(riga.get(campo, '')) != str(run[campo]):
            p.append(campo + '=' + repr(riga.get(campo)) + ' invece di ' + repr(run[campo]))
    # D12: la verita' di questo mondo deve essere SCESA sotto 18,5 (beta del
    # mondo base), e il costo del disegno deve essere positivo. Se beta fosse
    # stato ricalibrato, la quota vera tornerebbe 18,5 e il run non misurerebbe
    # niente: questo controllo lo intercetta riga per riga.
    if run['disegno'] == 'blackout':
        _qb = _numero(riga.get('quota_media_vera_base_pct'))
        if _qb is None or abs(_qb - 18.5) > 0.001:
            p.append('quota vera del mondo base=' + str(riga.get('quota_media_vera_base_pct')) +
                     ' invece di 18,5: beta non viene dal mondo base')
        _qv = _numero(riga.get('quota_media_vera_pct'))
        if _qv is None or not (17.6 <= _qv <= 18.3):
            p.append('quota vera di questo mondo=' + str(riga.get('quota_media_vera_pct')) +
                     ' fuori da 17,6-18,3: con beta fisso la verita' + A + ' deve scendere sotto 18,5')
        _pe = _numero(riga.get('candidature_vere_perse'))
        if _pe is None or _pe <= 0:
            p.append('candidature vere perse=' + str(riga.get('candidature_vere_perse')) +
                     ': il blackout non ha tolto niente')
    # D13: il livello di f e la presenza effettiva del controllo nel modello.
    if run.get('controllo_baseline'):
        _f = _numero(riga.get('controllo_baseline_f'))
        if _f is None or abs(_f - float(run['controllo_baseline'])) > 1e-9:
            p.append('controllo_baseline_f=' + str(riga.get('controllo_baseline_f')) +
                     ' invece di ' + str(run['controllo_baseline']))
        _cn = str(riga.get('controlli_nel_modello') or '')
        if 'ctrl_domanda_organica_osservata' not in _cn:
            p.append('il controllo di D13 non e' + A + ' fra i controlli del modello: ' + _cn)
        if str(riga.get('controllo_baseline_colonna')) != 'domanda_organica_osservata':
            p.append('colonna del controllo=' + str(riga.get('controllo_baseline_colonna')))
    imp = str(riga.get('impronta_input_data'))
    if run['impronta_attesa']:
        if imp != run['impronta_attesa']:
            p.append('impronta=' + imp + ' invece di ' + run['impronta_attesa'] +
                     ' (la spesa e' + A + ' cambiata: non e' + A + ' quel dataset)')
    elif run['nome'] in impronte and impronte[run['nome']] != imp:
        p.append('impronta=' + imp + ' invece di ' + impronte[run['nome']] +
                 ' registrata al primo fit di questo run: il mondo non e' + A + ' piu' + A +
                 ' lo stesso')
    if str(riga.get('unita_outcome')) != 'EUR per EUR speso':
        p.append('unita_outcome=' + str(riga.get('unita_outcome')))
    if not (_numero(riga.get('vero')) or 0) > 0:
        p.append('vero non positivo: ' + str(riga.get('vero')))
    draw = os.path.join(affiancato(csv_di(run), '_draw'), run['nome'] + '_draw.csv.gz')
    if not (os.path.exists(draw) and os.path.getsize(draw) > 0):
        p.append('file dei draw assente: ' + draw)
    return p


def righe_canali(run):
    righe = [r for r in leggi_csv(affiancato(csv_di(run), '_canali.csv')) if r.get('mondo') == run['nome']]
    return righe[-7:] if righe else []


def controllo_canali(run):
    """Le 7 righe per canale dell'ULTIMA esecuzione: 7 canali distinti, canali
    trattati come da pre-registrazione, dose per canale come da tabella
    (tolleranza 0,05 punti), riduzione effettiva come attesa."""
    tutte = [r for r in leggi_csv(affiancato(csv_di(run), '_canali.csv')) if r.get('mondo') == run['nome']]
    if not tutte or len(tutte) % 7 != 0:
        return ['righe per canale: ' + str(len(tutte)) + ' invece di 7'], ''
    ultime = tutte[-7:]
    p = []
    if len({r.get('canale') for r in ultime}) != 7:
        return ['le ultime 7 righe per canale non sono 7 canali distinti'], ''
    trattati = sorted(r['canale'] for r in ultime if str(r.get('trattato')) == '1')
    attesi = sorted(run['canali_trattati'].split(';')) if run['canali_trattati'] else []
    if trattati != attesi:
        p.append('canali trattati nel file canali: ' + ';'.join(trattati) + ' invece di ' + ';'.join(attesi))
    for r in ultime:
        ch = r['canale']
        if ch in run['dose_per_canale']:
            d = _numero(r.get('dose_pct'))
            if d is None or abs(d - float(run['dose_per_canale'][ch])) > 0.05:
                p.append('dose ' + ch + '=' + str(r.get('dose_pct')) + ' invece di ' +
                         str(run['dose_per_canale'][ch]))
            if run.get('riduzione_attesa') is not None:
                rid = _numero(r.get('riduzione_effettiva_pct'))
                if rid is None or abs(rid - float(run['riduzione_attesa'])) > 0.05:
                    p.append('riduzione effettiva ' + ch + '=' + str(r.get('riduzione_effettiva_pct')) +
                             ' invece di ' + str(run['riduzione_attesa']))
            elif 'intensita' in run['parametri_disegno']:
                rid = _numero(r.get('riduzione_effettiva_pct'))
                if rid is None or not (-99.0 < rid < 0.0):
                    p.append('riduzione effettiva ' + ch + '=' + str(r.get('riduzione_effettiva_pct')) +
                             ' non e' + A + ' una riduzione parziale')
    nota = ('canali: ' + str(len(tutte)) + ' righe per questo run (scritto ' + str(len(tutte) // 7) +
            ' volte, valgono le ultime 7)') if len(tutte) > 7 else ''
    return p, nota


def scostamento(run, ottenuto):
    """Regola del run. replica_4dec: il mondo 42 di un disegno gia' nel registro
    (dati a 3 decimali); qui i dati sono a 4: atteso -1..-1,7%, fuori = segnalato,
    oltre +/-2% = anomalia. prima_misura: nessun atteso sul numero."""
    if run['regola'] == 'prima_misura':
        return ('prima_misura', None, None, 'prima misura: nessun atteso sul numero (collaudo di sola configurazione)')
    if run['regola'] == 'identico_osservativo':
        # D0: la pipeline e' deterministica (run 1 e 5 del registro bit-identici):
        # atteso IDENTICO alla riga dello studio osservativo alla quarta cifra.
        atteso = float(run['atteso'])
        dev = 100.0 * (ottenuto / atteso - 1.0)
        if abs(ottenuto - atteso) <= 0.00005 + 1e-12:
            return ('ok', dev, atteso, 'IDENTICO allo studio osservativo (' + format(atteso, '.4f') + ')')
        return ('anomalia', dev, atteso, 'rapporto_mediana ' + format(ottenuto, '.4f') + ' invece di ' +
                format(atteso, '.4f') + ' dello studio osservativo (' + format(dev, '+.3f') +
                '%): la pipeline non e' + A + ' piu' + A + ' quella, anomalia')
    atteso = float(run['atteso'])
    dev = 100.0 * (ottenuto / atteso - 1.0)
    lo, hi = float(run['anomalia_min_pct']), float(run['anomalia_max_pct'])
    blo, bhi = float(run['banda_min_pct']), float(run['banda_max_pct'])
    if dev > hi + 1e-9:
        return ('anomalia', dev, atteso, 'scostamento ' + format(dev, '+.2f') + '% SOPRA la soglia ' +
                format(hi, '+.2f') + '% dal ' + format(atteso, '.2f') + ' del registro (3 decimali)')
    if dev < lo - 1e-9:
        return ('anomalia', dev, atteso, 'scostamento ' + format(dev, '+.2f') + '% SOTTO la soglia ' +
                format(lo, '+.2f') + '% dal ' + format(atteso, '.2f') + ' del registro')
    if blo - 1e-9 <= dev <= bhi + 1e-9:
        return ('ok', dev, atteso, 'scostamento ' + format(dev, '+.2f') + '% nella banda attesa ' +
                format(blo, '+.2f') + '..' + format(bhi, '+.2f') + '% (dati a 4 decimali contro 3)')
    return ('fuori_banda', dev, atteso, 'scostamento ' + format(dev, '+.2f') + '% fuori dalla banda ' +
            format(blo, '+.2f') + '..' + format(bhi, '+.2f') + '% ma dentro le soglie: segnalato')


def verifica(run, riga, impronte=None):
    impronte = leggi_impronte() if impronte is None else impronte
    v = {'n': run['n'], 'codice': run['codice'], 'nome': run['nome'], 'seed_mondo': run['seed_mondo'],
         'regola': run['regola'], 'atteso': '' if run['atteso'] is None else run['atteso'],
         'ottenuto': riga.get('rapporto_mediana') if riga else '',
         'pit': riga.get('pit') if riga else '', 'scostamento_pct': '', 'esito_scostamento': '',
         'ammissibile': '', 'divergenze_pct': '', 'problemi_config': '', 'esito': '', 'messaggio': ''}
    if riga is None:
        v['esito'] = 'nessuna_riga'
        v['messaggio'] = 'il runner non ha scritto nessuna riga (processo morto)'
        return v
    problemi = controllo_config(run, riga, impronte)
    nota_canali = ''
    if riga.get('esito') == 'ok':
        _pc, nota_canali = controllo_canali(run)
        problemi += _pc
    v['problemi_config'] = ' | '.join(problemi)
    if riga.get('esito') != 'ok':
        v['esito'] = 'errore_tecnico'
        v['messaggio'] = problemi[0] if problemi else 'esito non ok'
        return v
    div = _numero(riga.get('divergenze'))
    tot = (_numero(riga.get('n_chains')) or 0) * (_numero(riga.get('keep')) or 0)
    if div is not None and tot > 0:
        v['divergenze_pct'] = round(100.0 * div / tot, 3)
    # Criterio di ammissibilita' (cap. 4.7): R-hat <= 1,1 ed ESS >= 100 su roi_m
    # e beta_m. Chi non passa e' riportato col suo numero e MAI rifatto.
    _amm = []
    for _campo, _soglia, _verso in (('rhat_roi_m', 1.1, 'max'), ('rhat_beta_m', 1.1, 'max'),
                                    ('ess_roi_m', 100.0, 'min'), ('ess_beta_m', 100.0, 'min')):
        _val = _numero(riga.get(_campo))
        if _val is None:
            _amm.append(_campo + ' assente')
        elif _verso == 'max' and _val > _soglia + 1e-9:
            _amm.append(_campo + '=' + str(_val) + ' > ' + str(_soglia))
        elif _verso == 'min' and _val < _soglia:
            _amm.append(_campo + '=' + str(_val) + ' < ' + str(int(_soglia)))
    v['ammissibile'] = 'no' if _amm else 'si'
    ottenuto = _numero(riga.get('rapporto_mediana'))
    if ottenuto is None:
        problemi.append('rapporto_mediana non numerico: ' + str(riga.get('rapporto_mediana')))
    else:
        es, dev, usato, msg = scostamento(run, ottenuto)
        v['esito_scostamento'] = es
        v['scostamento_pct'] = '' if dev is None else round(dev, 3)
        v['messaggio'] = msg
        if run['regola'] == 'identico_osservativo' and es == 'ok':
            oss = run['osservativo_atteso']
            for campo, dec in (('pit', 6), ('rapporto_ci05', 4), ('rapporto_ci95', 4), ('dentro', 0)):
                ott, att = _numero(riga.get(campo)), _numero(oss.get(campo))
                if ott is None or att is None or abs(round(ott, dec) - round(att, dec)) > 1e-9:
                    problemi.append(campo + '=' + str(riga.get(campo)) + ' invece di ' + str(oss.get(campo)) +
                                    ' dello studio osservativo')
    v['problemi_config'] = ' | '.join(problemi)      # anche i problemi trovati dopo (D0: pit, forbice)
    if problemi:
        v['esito'] = 'anomalia'
        v['messaggio'] = ('configurazione o disegno diversi dalla tabella: ' + ' | '.join(problemi) +
                          ((' ; ' + v['messaggio']) if v['messaggio'] else ''))
    elif _amm:
        v['esito'] = 'non_ammissibile'
        v['messaggio'] = ('NON AMMISSIBILE (cap. 4.7): ' + ', '.join(_amm) +
                          ' ; numero riportato, fuori dai conteggi principali')
    elif v['esito_scostamento'] == 'anomalia':
        v['esito'] = 'anomalia'
    elif v['esito_scostamento'] == 'fuori_banda':
        v['esito'] = 'segnalato'
    else:
        v['esito'] = 'ok'
    if v['divergenze_pct'] not in ('', None) and float(v['divergenze_pct']) > 1.0:
        v['messaggio'] += ' ; divergenze ' + str(v['divergenze_pct']) + '% > 1%'
    if nota_canali:
        v['messaggio'] += ' ; ' + nota_canali
    return v


# ============================================================================
# PRE-VOLO - otto controlli
# ============================================================================
_ESITI = []
_RIMEDI = []


def _registra(numero, titolo, ok, dettaglio, rimedio=''):
    _ESITI.append((numero, titolo, bool(ok), dettaglio))
    if not ok:
        _RIMEDI.append(str(numero) + '. ' + titolo + '\n   ' + dettaglio +
                       ('\n   COSA FARE: ' + rimedio if rimedio else ''))
    print(('   [ok] ' if ok else '   [NO] ') + dettaglio)
    return bool(ok)


print('PRE-VOLO della griglia sperimentale')
print('=' * 72)

# --- 1. la GPU ---------------------------------------------------------------
print(' 1. GPU')
_nome_gpu = ''
try:
    _r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                        capture_output=True, text=True, timeout=120)
    if _r.returncode == 0 and _r.stdout.strip():
        _nome_gpu = _r.stdout.strip().splitlines()[0].strip()
except (OSError, subprocess.SubprocessError):
    _nome_gpu = ''
if not _nome_gpu:
    _ok1 = _registra(1, 'GPU', False, 'nessuna GPU collegata a questo runtime',
                     'Runtime -> Cambia tipo di runtime -> Acceleratore hardware: GPU (T4) -> '
                     'Salva, poi Esegui tutto di nuovo.')
elif 'T4' in _nome_gpu.upper():
    _ok1 = _registra(1, 'GPU', True, _nome_gpu)
else:
    _ok1 = _registra(1, 'GPU', True, _nome_gpu + ' (non e' + A + ' una T4: va bene, tempi diversi)')

# --- 2. Drive e la cartella NUOVA --------------------------------------------
print(' 2. Google Drive (autorizza il popup: e' + A + ' l' + A + 'unica cosa da fare)')
_ok2 = False
try:
    from google.colab import drive as _drive
except ImportError:
    _ok2 = _registra(2, 'Drive', False, 'non sei su Colab (google.colab non importabile)',
                     'Questo notebook e' + A + ' fatto per Google Colab.')
else:
    try:
        _drive.mount('/content/drive')
        _montato = os.path.isdir('/content/drive/MyDrive')
        _errore_mount = ''
    except Exception as _e:
        _montato = False
        _errore_mount = str(_e)[:120]
    if not _montato:
        _ok2 = _registra(2, 'Drive', False,
                         'Drive non montato' + (': ' + _errore_mount if _errore_mount else ''),
                         'Riesegui la cella e clicca "Connetti a Google Drive".')
    else:
        os.makedirs(DIR_PREREG, exist_ok=True)
        _ok2 = _registra(2, 'Drive', True, CARTELLA_DRIVE + ' pronta (cartella nuova; '
                         'MMM_copertura e MMM_registro_run non vengono toccate)')

# --- 3. anomalie di lanci precedenti (non bloccano) --------------------------
print(' 3. anomalie di lanci precedenti (non bloccano)')
if not _ok2:
    _ok3 = _registra(3, 'anomalie', False, 'non eseguito: dipende dal controllo 2 (Drive)')
else:
    _impronte0 = leggi_impronte()
    _presenti = sorted(f for f in os.listdir(CARTELLA_DRIVE)
                       if f.startswith('ANOMALIA_') and f.endswith('.txt'))
    _restano, _risolti = [], []
    for _f_an in _presenti:
        _nome_an = _f_an[len('ANOMALIA_'):-len('.txt')]
        _run_an = RUN_PER_NOME.get(_nome_an)
        _righe_an = ([r for r in leggi_csv(csv_di(_run_an)) if r.get('mondo') == _nome_an]
                     if _run_an else [])
        _v_an = verifica(_run_an, _righe_an[-1], _impronte0) if (_run_an and _righe_an) else None
        if _v_an is not None and _v_an['esito'] != 'anomalia':
            _vecchio = os.path.join(CARTELLA_DRIVE, _f_an)
            _nuovo = os.path.join(CARTELLA_DRIVE, 'RISOLTO_' + _nome_an + '.txt')
            with open(_vecchio, 'a', encoding='utf-8') as _fs:
                _fs.write('\n---\nRIESAMINATO il ' + time.strftime('%Y-%m-%d %H:%M:%S') +
                          ' con gli attesi versione ' + str(VERSIONE_ATTESI) + ': esito ' +
                          _v_an['esito'] + ' - ' + str(_v_an['messaggio']) + '\n')
            os.replace(_vecchio, _nuovo)
            _risolti.append(_f_an)
            print('      ' + _f_an + ' -> RISOLTO: ' + _v_an['esito'] + ' - ' + str(_v_an['messaggio'])[:110])
        else:
            _restano.append(_f_an)
            print('      ' + _f_an + ': resta un' + A + 'anomalia (' +
                  (str(_v_an['messaggio'])[:100] if _v_an else 'riga assente') +
                  '). Il ciclo prosegue lo stesso; si sistema al ritorno.')
    _ok3 = _registra(3, 'anomalie', True,
                     ('nessuna anomalia registrata' if not _presenti else
                      str(len(_restano)) + ' anomalia/e restano come promemoria, ' +
                      str(len(_risolti)) + ' risolta/e; nessuna blocca il ciclo'))

# --- 4. pre-registrazioni e attesi su Drive, PRIMA di qualunque fit -----------
print(' 4. pre-registrazioni e attesi su Drive')
if not _ok2:
    _ok4 = _registra(4, 'pre-registrazioni', False, 'non eseguito: dipende dal controllo 2 (Drive)')
else:
    _note4 = []
    _norm = lambda t: t.replace('\r\n', '\n')
    _file_attivi = sorted({INFO_DISEGNO[d]['prereg'] for d in DISEGNI_ATTIVI})
    for _nf in _file_attivi:
        _p = os.path.join(DIR_PREREG, _nf)
        _testo = PREREGISTRAZIONI[_nf]
        if not os.path.exists(_p):
            open(_p, 'w', encoding='utf-8', newline='\n').write(_testo)
            _note4.append(_nf + ' scritto')
        elif _norm(open(_p, encoding='utf-8').read()) != _norm(_testo):
            _cons = _p[:-3] + '_precedente_' + time.strftime('%Y%m%d_%H%M%S') + '.md'
            os.replace(_p, _cons)
            open(_p, 'w', encoding='utf-8', newline='\n').write(_testo)
            _note4.append('AVVISO: ' + _nf + ' su Drive era diverso: conservato come ' +
                          os.path.basename(_cons) + ', scritto quello di questo notebook')
        else:
            _note4.append(_nf + ' gia' + A + ' su Drive, identico')
    if not os.path.exists(ATTESI_DRIVE):
        open(ATTESI_DRIVE, 'w', encoding='utf-8', newline='').write(ATTESI_CSV_TESTO)
        _note4.append('attesi_griglia.csv scritto (' + str(len(TABELLA)) + ' run)')
    elif _norm(open(ATTESI_DRIVE, encoding='utf-8', newline='').read()) != _norm(ATTESI_CSV_TESTO):
        _cons = ATTESI_DRIVE[:-4] + '_precedente_' + time.strftime('%Y%m%d_%H%M%S') + '.csv'
        os.replace(ATTESI_DRIVE, _cons)
        open(ATTESI_DRIVE, 'w', encoding='utf-8', newline='').write(ATTESI_CSV_TESTO)
        _note4.append('AVVISO: attesi_griglia.csv su Drive era diverso: conservato come ' +
                      os.path.basename(_cons) + ', scritto quello di questo notebook (v' +
                      str(VERSIONE_ATTESI) + '); le righe gia' + A + ' registrate vengono rilette')
    else:
        _note4.append('attesi_griglia.csv gia' + A + ' su Drive, identico (v' + str(VERSIONE_ATTESI) + ')')
    for _n in _note4:
        print('      ' + _n)
    _ok4 = _registra(4, 'pre-registrazioni', True, str(len(_file_attivi)) + ' pre-registrazioni e gli '
                     'attesi sono su Drive prima del primo fit' +
                     (' (con avvisi, vedi sopra)' if any(n.startswith('AVVISO') for n in _note4) else ''))

# --- 5. repo, generatore ricostruito da main + patch, runner -------------------
print(' 5. repo, patch del generatore (A + G1 + G2 + G3 + G4 + G6), runner R8')
_ok5 = False
try:
    if os.path.isdir(REPO) and not os.path.isdir(os.path.join(REPO, '.git')):
        shutil.rmtree(REPO)
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', URL_REPO, REPO], check=True)
    else:
        print('      repo gia' + A + ' presente, non riclonato')
    _commit = subprocess.run(['git', '-C', REPO, 'log', '-1', '--oneline'],
                             capture_output=True, text=True).stdout.strip()
    print('      commit: ' + _commit)
    _p = subprocess.run(['git', '-C', REPO, 'show', 'HEAD:genera_dati_simulati.py'], capture_output=True)
    if _p.returncode != 0:
        raise RuntimeError('non riesco a rileggere genera_dati_simulati.py da git: ' +
                           _p.stderr.decode('utf-8', 'replace')[-300:])
    _src = _p.stdout.decode('utf-8')
    for _k, (_vecchio, _nuovo) in enumerate(_dati['patches_generatore']):
        if _src.count(_vecchio) != 1:
            raise RuntimeError('patch A del generatore NON applicabile alla sostituzione ' + str(_k + 1) +
                               ': frammento trovato ' + str(_src.count(_vecchio)) + ' volte. Il generatore '
                               'su main e' + A + ' cambiato da quando questo notebook e' + A + ' stato preparato')
        _src = _src.replace(_vecchio, _nuovo)
    if shutil.which('patch') is None:
        raise RuntimeError('il comando `patch` non e' + A + ' disponibile: serve per applicare G1/G2/G3')
    _tmp = tempfile.mkdtemp()
    _f_gen = os.path.join(_tmp, 'genera_dati_simulati.py')
    open(_f_gen, 'w', encoding='utf-8', newline='\n').write(_src)
    for _nome_diff in ('G1', 'G2', 'G3', 'G4', 'G6'):
        _f_diff = os.path.join(_tmp, _nome_diff + '.diff')
        open(_f_diff, 'w', encoding='utf-8', newline='\n').write(_dati['diff_generatore'][_nome_diff])
        _r = subprocess.run(['patch', '-p0', '-F0', '--batch', '--no-backup-if-mismatch',
                             'genera_dati_simulati.py', '-i', _f_diff],
                            cwd=_tmp, capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError('patch ' + _nome_diff + ' NON applicabile a fuzz 0: ' +
                               (_r.stdout + _r.stderr).strip()[-300:])
    _src = open(_f_gen, encoding='utf-8').read()
    _sha = hashlib.sha256(_src.encode('utf-8')).hexdigest()[:16]
    if _sha != _dati['generatore_sha']:
        raise RuntimeError('il generatore ricostruito da main + A + G1..G4 + G6 ha impronta ' + _sha +
                           ', il collaudo era su ' + _dati['generatore_sha'] + ': non e' + A +
                           ' lo stesso file')
    compile(_src, GEN, 'exec')
    open(GEN, 'w', encoding='utf-8', newline='\n').write(_src)
    print('      generatore: ' + str(len(_dati['patches_generatore'])) + ' sostituzioni A + 5 diff a fuzz 0 '
          'dal file committato; sha ' + _sha + ' = collaudato')
    compile(_dati['runner'], RUNNER, 'exec')
    open(RUNNER, 'w', encoding='utf-8', newline='\n').write(_dati['runner'])
    print('      runner R8 scritto: ' + RUNNER)

    def _aiuto(percorso, etichetta):
        r = subprocess.run([sys.executable, percorso, '--help'], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(etichetta + ' non parte nemmeno per l' + A + 'help: ' +
                               ((r.stderr or r.stdout) or '').strip()[-400:])
        return r.stdout
    _h = _aiuto(GEN, 'il generatore')
    for _flag in ('--seed-mondo', '--sanita', '--geo', '--geo-canali', '--geo-regioni', '--geo-intensita',
                  '--geo-sfalsato', '--calendario', '--calendario-rotazione', '--calendario-giri',
                  '--casuale2', '--suffisso-mondo', '--blackout'):
        if _flag not in _h:
            raise RuntimeError('autotest: ' + _flag + ' assente dal generatore')
    _h = _aiuto(RUNNER, 'il runner')
    for _flag in ('--disegno', '--parametri-disegno', '--knots-per-quarter', '--prior', '--prior-mu',
                  '--prior-sigma', 'ancorato-benchmark', '--nome-run', '--controllo-baseline',
                  'blackout'):
        if _flag not in _h:
            raise RuntimeError('autotest: ' + _flag + ' assente dal runner')
    for _pezzo in ('def leggi_disegno', 'def parametri_disegno', 'spesa_ridistribuita_eur',
                   'def roas_benchmark', 'prior_mu_per_canale', 'def scrivi_controllo_baseline',
                   'def leggi_costo_blackout', 'spesa_tolta_eur',
                   'REVENUE_PER_KPI_PRESENTE', 'stdin=subprocess.DEVNULL'):
        if _pezzo not in _dati['runner']:
            raise RuntimeError('autotest: nel runner manca ' + _pezzo)
    _ok5 = _registra(5, 'repo e patch', True, 'generatore = main + A + G1..G4 + G6 (sha verificato), '
                     'runner R8 (--disegno blackout, --controllo-baseline); autotest passato')
except Exception as _e:
    _ok5 = _registra(5, 'repo e patch', False, str(_e)[:400],
                     'Se e' + A + ' il clone: rete, rilancia. Se manca `patch`: in una cella '
                     '`!apt-get install -y patch`. Se una patch non si applica: il generatore su main '
                     'e' + A + ' cambiato, serve un notebook aggiornato; riportalo in chat.')

# --- 6. cancello di fedelta': i cinque dataset del mondo 42 ------------------
print(' 6. cancello di fedelta' + A + ' (mondo 42: base, sanita, pause2, casuale2, geo contro i committati)')


def _uguale_numerico(a, b):
    ra = list(csv.reader(open(a, newline='', encoding='utf-8')))
    rb = list(csv.reader(open(b, newline='', encoding='utf-8')))
    if len(ra) != len(rb):
        return False
    for x, y in zip(ra, rb):
        if len(x) != len(y):
            return False
        for u, v in zip(x, y):
            if u == v:
                continue
            try:
                if abs(float(u) - float(v)) <= 1e-9:
                    continue
            except ValueError:
                pass
            return False
    return True


def _decimali(percorso):
    minimo = None
    with open(percorso, newline='', encoding='utf-8') as f:
        for r in csv.DictReader(f):
            v = str(r.get('indice_stagionale', '')).strip()
            d = len(v.split('.')[1]) if '.' in v else 0
            minimo = d if minimo is None else min(minimo, d)
    return minimo


_CASI_FEDELTA = [
    ('base', [], 'mondo_0042', 'dati_simulati', ''),
    ('sanita', ['--sanita'], 'mondo_0042_sanita', 'dati_simulati/sanita', '_sanita'),
    ('pause2', ['--pause2'], 'mondo_0042_pause2', 'dati_simulati/pause2', '_pause2'),
    ('casuale2', ['--casuale2'], 'mondo_0042_casuale2', 'dati_simulati/casuale2', '_casuale2'),
    ('geo', ['--geo'], 'mondo_0042_geo', 'dati_simulati/geo', '_geo'),
]


def _gate_disegno():
    """Cancello di fedelta' specifico di D12, sul mondo 42: le impronte dei
    sette file media del mondo blackout devono essere quelle dichiarate nella
    pre-registrazione (controllo 5), nelle 80 celle trattate non deve esserci
    NESSUNA riga (spesa zero = nessuna riga, regola gia' della base), e il
    JSON deve dire che beta viene dal mondo base (quota vera del mondo base
    18,5%) e quanto il disegno e' costato."""
    _g = subprocess.run([sys.executable, GEN, '--seed-mondo', '42', '--blackout'],
                        cwd=REPO, capture_output=True, text=True, stdin=subprocess.DEVNULL)
    if _g.returncode != 0:
        raise RuntimeError('generatore --blackout fallito: ' + (_g.stderr or _g.stdout)[-300:])
    _m = os.path.join(REPO, 'dati_simulati_mondi', 'mondo_0042_blackout', 'modello')
    _v = os.path.join(REPO, 'dati_simulati_mondi', 'mondo_0042_blackout', 'verita')
    _nomi = ['google_ads_settimanale.csv', 'meta_ads_settimanale.csv',
             'linkedin_ads_settimanale.csv', 'indeed_settimanale.csv',
             'subito_lavoro_settimanale.csv', 'jooble_settimanale.csv',
             'altre_job_board_settimanale.csv']
    _h = hashlib.sha256()
    for _n in _nomi:
        # NORMALIZZATO A LF: il fine riga dipende dal sistema operativo (CRLF su
        # Windows, LF su Linux) e non e' un dato. Senza questa riga l'impronta
        # calcolata su Windows non torna su Colab: e' successo il 21/9/2026 e
        # il cancello si e' fermato, correttamente, su una differenza che non
        # era nei numeri.
        _h.update(open(os.path.join(_m, _n), 'rb').read().replace(bytes([13, 10]), bytes([10])))
    _imp = _h.hexdigest()[:16]
    _atteso = _dati['impronte_media_attese']['42']['blackout']
    if _imp != _atteso:
        raise RuntimeError('impronta dei file media del mondo 42 blackout ' + _imp +
                           ' invece di ' + _atteso + ' della pre-registrazione')
    _j = json.load(open(os.path.join(_v, 'parametri_generazione.json'), encoding='utf-8'))
    _sc = _j.get('scenario_blackout') or {}
    if not _sc:
        raise RuntimeError('il JSON non ha scenario_blackout: il generatore non e' + A + ' quello con G6')
    if abs(float(_sc['quota_media_vera_mondo_base_pct']) - 18.5) > 1e-9:
        raise RuntimeError('quota vera del mondo base ' + str(_sc['quota_media_vera_mondo_base_pct']) +
                           ' invece di 18,5: beta non e' + A + ' quello del mondo base')
    if not 17.5 <= float(_sc['quota_media_vera_di_questo_mondo_pct']) < 18.5:
        raise RuntimeError('quota vera del mondo blackout ' +
                           str(_sc['quota_media_vera_di_questo_mondo_pct']) +
                           ': con beta fisso deve SCENDERE sotto 18,5')
    _reg = set(_sc['regioni_trattate'])
    _bl = [tuple(b) for b in _sc['blocchi_1based_inclusivi']]
    _sett = sorted({r['settimana'] for r in csv.DictReader(
        open(os.path.join(_m, 'candidature_settimanali.csv'), newline='', encoding='utf-8'))})
    _tratt = set()
    for _a, _b in _bl:
        _tratt |= set(_sett[_a - 1:_b])
    _righe = 0
    for _n in _nomi:
        with open(os.path.join(_m, _n), newline='', encoding='utf-8') as _f:
            for _r in csv.DictReader(_f):
                if _r['settimana'] in _tratt and _r['regione'] in _reg:
                    _righe += 1
    if _righe:
        raise RuntimeError(str(_righe) + ' righe di spesa nelle celle trattate: il blackout non e' + A +
                           ' totale')
    _costo = _sc['costo_del_disegno']
    shutil.rmtree(os.path.join(REPO, 'dati_simulati_mondi', 'mondo_0042_blackout'),
                  ignore_errors=True)
    return ('mondo 42 blackout: impronta ' + _imp + ' = pre-registrazione; 0 righe nelle 80 celle '
            'trattate; quota vera ' + str(_sc['quota_media_vera_di_questo_mondo_pct']) + '% (base 18,5%); '
            'costo ' + str(_costo['candidature_vere_perse']) + ' candidature vere e ' +
            str(_sc['spesa_tolta_totale_eur']) + ' EUR di spesa')


_ok6 = False
if not _ok5:
    _ok6 = _registra(6, 'fedelta' + A, False, 'non eseguito: dipende dal controllo 5 (repo)')
else:
    _problemi6 = []
    for _nome6, _flag6, _cart6, _ref6, _suf6 in _CASI_FEDELTA:
        try:
            _g = subprocess.run([sys.executable, GEN, '--seed-mondo', '42'] + _flag6, cwd=REPO,
                                capture_output=True, text=True, stdin=subprocess.DEVNULL)
            if _g.returncode != 0:
                raise RuntimeError('generatore fallito: ' + (_g.stderr or _g.stdout)[-300:])
            _mondo = os.path.join(REPO, 'dati_simulati_mondi', _cart6)
            _mprod = os.path.join(_mondo, 'modello')
            _mref = os.path.join(REPO, _ref6, 'modello' + _suf6)
            _diversi, _formato = [], []
            for _nf in sorted(os.listdir(_mref)):
                _a, _b = os.path.join(_mref, _nf), os.path.join(_mprod, _nf)
                if not os.path.exists(_b):
                    _diversi.append(_nf + ' (assente)')
                elif open(_a, 'rb').read() == open(_b, 'rb').read():
                    continue
                elif _nf == 'stagionalita.csv' and _uguale_numerico(_a, _b):
                    _formato.append(_nf)
                else:
                    _diversi.append(_nf)
            _va = os.path.join(REPO, _ref6, 'verita', 'contributo_vero' + _suf6 + '.csv')
            _vb = os.path.join(_mondo, 'verita', 'contributo_vero.csv')
            if not _uguale_numerico(_va, _vb):
                _diversi.append('contributo_vero')
            _dec = _decimali(os.path.join(_mprod, 'stagionalita.csv'))
            if _dec != int(CONFIG['decimali']):
                _diversi.append('stagionalita.csv a ' + str(_dec) + ' decimali invece di ' + str(CONFIG['decimali']))
            if _diversi:
                raise RuntimeError('differisce dal committato in ' + ', '.join(_diversi))
            print('      ' + _nome6.ljust(9) + ' identico al committato (' + ', '.join(_formato or ['nessun file']) +
                  ' solo nel formato a 4 decimali)')
            shutil.rmtree(_mondo, ignore_errors=True)
        except Exception as _e:
            _problemi6.append(_nome6 + ': ' + str(_e)[:200])
            print('      ' + _nome6.ljust(9) + ' ERRORE: ' + str(_e)[:160])
    _nota_disegno = ''
    if not _problemi6:
        try:
            _nota_disegno = _gate_disegno()
            print('      D12       ' + _nota_disegno)
        except Exception as _e:
            _problemi6.append('D12: ' + str(_e)[:250])
            print('      D12       ERRORE: ' + str(_e)[:200])
    if _problemi6:
        _ok6 = _registra(6, 'fedelta' + A, False, '; '.join(_problemi6)[:500],
                         'Il generatore patchato non riproduce piu' + A + ' i dataset della tesi, '
                         'oppure il disegno nuovo non e' + A + ' quello dichiarato: NESSUN fit parte. '
                         'Riportalo in chat.')
    else:
        _ok6 = _registra(6, 'fedelta' + A, True, '5 dataset del mondo 42 rigenerati identici ai committati '
                         '(base, sanita, pause2, casuale2, geo); percorso senza flag invariato; ' +
                         _nota_disegno)

# --- 7. i semi contro lo studio osservativo -----------------------------------
print(' 7. semi dei mondi contro copertura_risultati.csv (studio osservativo)')
_ok7 = False
if not _ok2:
    _ok7 = _registra(7, 'semi', False, 'non eseguito: dipende dal controllo 2 (Drive)')
else:
    _f_cop = os.path.join(CARTELLA_COPERTURA, 'copertura_risultati.csv')
    if os.path.exists(_f_cop):
        _righe_cop = {int(float(r['seed_mondo'])): r for r in leggi_csv(_f_cop) if _numero(r.get('seed_mondo')) is not None}
        _mancanti = [m for m in MONDI_ATTIVI if m not in _righe_cop]
        # le righe dello studio osservativo su Drive devono essere quelle con cui e'
        # stata scritta la pre-registrazione di D0 (incorporate nel notebook)
        _diverse = []
        for _m in MONDI_ATTIVI:
            if _m in _righe_cop and _m in OSSERVATIVO:
                for _c in ('rapporto_mediana', 'pit', 'impronta_input_data', 'rapporto_ci05', 'rapporto_ci95'):
                    if str(_righe_cop[_m].get(_c, '')).strip() != str(OSSERVATIVO[_m].get(_c, '')).strip():
                        _diverse.append(str(_m) + ':' + _c)
        if _mancanti:
            _ok7 = _registra(7, 'semi', False, 'semi ' + str(_mancanti) + ' non sono nello studio osservativo (' +
                             str(sorted(_righe_cop)) + ')', 'I mondi della griglia devono essere gli otto dello studio '
                             'osservativo: MONDI_ATTIVI va corretto, non il CSV.')
        elif _diverse:
            _ok7 = _registra(7, 'semi', False, 'copertura_risultati.csv su Drive NON coincide con le righe '
                             'incorporate nella pre-registrazione di D0: ' + ', '.join(_diverse)[:200],
                             'Il file su Drive deve essere quello dello studio osservativo del 13/9 (8 mondi, '
                             '0 su 8). Se e' + A + ' stato sovrascritto, ripristinalo; non si aggiustano gli attesi.')
        else:
            _ok7 = _registra(7, 'semi', True, 'mondi ' + str(MONDI_ATTIVI) + ' presenti in ' + _f_cop +
                             ', righe identiche a quelle incorporate (attese di D0)')
    else:
        _ok7 = _registra(7, 'semi', False, _f_cop + ' non trovato: i semi non possono essere verificati',
                         'Copia copertura_risultati.csv dello studio osservativo (8 mondi) in ' + CARTELLA_COPERTURA +
                         ' e rilancia. Era la precondizione dichiarata nella pre-registrazione randomizzata: '
                         'non si salta.')

# --- 8. Meridian esattamente 1.8.0 -------------------------------------------
print(' 8. Meridian ' + VERSIONE_MERIDIAN + ' esatta')
_CODICE_VERSIONE = (
    'import sys\n'
    'try:\n'
    '    import importlib.metadata as _md\n'
    '    d = _md.version("google-meridian")\n'
    'except Exception:\n'
    '    d = "assente"\n'
    'try:\n'
    '    import meridian\n'
    '    i = meridian.__version__\n'
    'except Exception as e:\n'
    '    i = "NON IMPORTABILE (" + type(e).__name__ + ": " + str(e)[:100] + ")"\n'
    'sys.stdout.write(d + "|" + i)\n')


def _versione_meridian():
    r = subprocess.run([sys.executable, '-c', _CODICE_VERSIONE], capture_output=True, text=True)
    testo = (r.stdout or '').strip()
    if '|' not in testo:
        return 'assente', 'NESSUN OUTPUT: ' + ((r.stderr or '').strip()[-160:] or '?')
    return testo.split('|', 1)


_TITOLO8 = 'Meridian ' + VERSIONE_MERIDIAN
_RIMEDIO8 = ('Esegui in una cella nuova:  !pip install -q google-meridian==' + VERSIONE_MERIDIAN +
             '   poi Runtime -> Riavvia sessione e Esegui tutto.')
if not (_ok1 and _ok2 and _ok3 and _ok4 and _ok5 and _ok6 and _ok7):
    _ok8 = _registra(8, _TITOLO8, False, 'non eseguito: prima vanno sistemati i controlli falliti qui sopra')
else:
    _dist, _imp = _versione_meridian()
    _errore_pip = ''
    if _dist != VERSIONE_MERIDIAN:
        print('      trovata "' + _dist + '": installo google-meridian==' + VERSIONE_MERIDIAN + '...')
        _ins = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                               'google-meridian==' + VERSIONE_MERIDIAN, 'openpyxl', 'statsmodels'],
                              capture_output=True, text=True)
        if _ins.returncode != 0:
            _errore_pip = (((_ins.stderr or _ins.stdout) or '').strip()[-200:] or 'senza messaggio')
        else:
            _dist, _imp = _versione_meridian()
    if _errore_pip:
        _ok8 = _registra(8, _TITOLO8, False, 'installazione fallita: ' + _errore_pip, _RIMEDIO8)
    elif _dist != VERSIONE_MERIDIAN:
        _ok8 = _registra(8, _TITOLO8, False, 'installata la versione "' + _dist + '"', _RIMEDIO8)
    elif _imp != VERSIONE_MERIDIAN:
        _ok8 = _registra(8, _TITOLO8, False, 'distribuzione ' + _dist + ' ma import KO: ' + _imp,
                         'Runtime -> Riavvia sessione, poi Esegui tutto.')
    else:
        _ok8 = _registra(8, _TITOLO8, True, VERSIONE_MERIDIAN + ' (distribuzione e import concordi)')

print()
print('=' * 72)
print('PRE-VOLO, otto controlli')
for _n, _titolo, _ok, _dettaglio in _ESITI:
    print(' ' + (SPUNTA if _ok else CROCE) + '  ' + str(_n) + '. ' + _titolo.ljust(20) + ' ' +
          _dettaglio.splitlines()[0][:150])
print('=' * 72)
if _RIMEDI:
    raise SystemExit('PRE-VOLO FALLITO su ' + str(len(_RIMEDI)) + ' controllo/i su 8. Il ciclo NON parte.\n\n' +
                     '\n\n'.join(_RIMEDI) + '\n\nSistemato il problema: Esegui tutto di nuovo.')
print('pre-volo superato: 8 su 8.')
print()
SEQUENZA = [r for r in TABELLA if r['codice'] in DISEGNI_ATTIVI and int(r['seed_mondo']) in MONDI_ATTIVI]
SEQUENZA.sort(key=lambda r: (DISEGNI_ATTIVI.index(r['codice']), int(r['n'])))
for _d in DISEGNI_ATTIVI:
    _righe_d = [r for r in SEQUENZA if r['codice'] == _d]
    _fatti_d = {r['mondo'] for r in leggi_csv(os.path.join(CARTELLA_DRIVE, INFO_DISEGNO[_d]['csv']))
                if r.get('esito') == 'ok'}
    print(' ' + _d + ' ' + INFO_DISEGNO[_d]['titolo'].ljust(70) + ' ' + str(len(_righe_d)).rjust(3) +
          ' run, gia' + A + ' fatti ' + str(len([r for r in _righe_d if r['nome'] in _fatti_d])))
SETUP_COMPLETO = True
print()
print('=' * 72)
print('SETUP COMPLETO. La cella successiva parte da sola: circa dieci minuti a fit,')
print(str(len(SEQUENZA)) + ' fit in tutto per i disegni attivi; il mondo 42 di ogni disegno per primo.')
print('=' * 72)


In [ ]:
# ============================================================================
# CELLA 2 - IL CICLO: un disegno per volta, mondo 42 per primo, controllo per riga
# ============================================================================
# Per ogni run: lancia il runner in un SOTTOPROCESSO (genera il mondo col suo
# disegno, fitta ~10 minuti, misura, accoda UNA riga nel CSV del disegno su
# Drive, libera la GPU), poi confronta la riga con la SUA riga della tabella:
# configurazione del fit, disegno (canali trattati, dose per canale, riduzione
# effettiva), impronta, ammissibilita'. Una differenza = ANOMALIA: si scrive
# ANOMALIA_<run>.txt e il ciclo PROSEGUE. Niente ferma il ciclo. Se Colab cade,
# si rilancia: riprende dal primo run non registrato con esito ok.
import csv, os, statistics, subprocess, sys, time

try:
    SETUP_COMPLETO
except NameError:
    raise SystemExit('Esegui prima la CELLA 1 (pre-volo e setup).')


def durata(secondi):
    s = int(secondi + 0.5)
    return (str(s // 3600) + 'h ' + format((s % 3600) // 60, '02d') + 'm ' + format(s % 60, '02d') + 's')


def righe_ok(percorso):
    out = {}
    for r in leggi_csv(percorso):
        if r.get('esito') == 'ok':
            out[r['mondo']] = r
    return out


def riga_del_run(percorso, nome):
    righe = [r for r in leggi_csv(percorso) if r.get('mondo') == nome]
    return righe[-1] if righe else None


def comando(run):
    c = [sys.executable, RUNNER, '--seed-mondo', str(run['seed_mondo']), '--disegno', run['disegno'],
         '--repo', REPO, '--csv', csv_di(run), '--keep', str(KEEP),
         '--knots-per-quarter', str(CONFIG['knots_per_quarter']),
         '--seed-mcmc', str(CONFIG['seed_mcmc']), '--prior', run['prior_tipo'],
         '--nome-run', run['nome']]
    if run['parametri_disegno']:
        c += ['--parametri-disegno', run['parametri_disegno']]
    # D13: il livello del controllo costruito dalla baseline vera. Assente o 0
    # = nessun file scritto, run identico a D0.
    if run.get('controllo_baseline'):
        c += ['--controllo-baseline', str(run['controllo_baseline'])]
    # prior parametrico (D9): i flag si passano solo quando differiscono dal
    # percorso canonico, cosi' i run canonici restano identici al registro
    if run['prior_tipo'] == 'riferimento':
        if run['prior_mu'] not in ('', None) and abs(float(run['prior_mu']) - 0.2) > 1e-12:
            c += ['--prior-mu', str(run['prior_mu'])]
        if abs(float(run['prior_sigma']) - 0.9) > 1e-12:
            c += ['--prior-sigma', str(run['prior_sigma'])]
    else:
        c += ['--prior-sigma', str(run['prior_sigma'])]
    return c


COLONNE_VERIFICHE = ['quando', 'n', 'codice', 'nome', 'seed_mondo', 'regola', 'atteso', 'ottenuto', 'pit',
                     'scostamento_pct', 'esito_scostamento', 'ammissibile', 'divergenze_pct',
                     'problemi_config', 'esito', 'messaggio']


def accoda_verifica(run, v):
    p = affiancato(csv_di(run), '_verifiche.csv')
    if os.path.exists(p) and os.path.getsize(p) > 0:
        with open(p, newline='', encoding='utf-8') as f:
            intestazione = next(csv.reader(f), [])
        if intestazione != COLONNE_VERIFICHE:
            vecchie = leggi_csv(p)
            with open(p, 'w', newline='', encoding='utf-8') as f:
                w = csv.DictWriter(f, fieldnames=COLONNE_VERIFICHE, extrasaction='ignore', restval='')
                w.writeheader()
                for r in vecchie:
                    w.writerow(r)
    nuovo = not (os.path.exists(p) and os.path.getsize(p) > 0)
    with open(p, 'a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=COLONNE_VERIFICHE, extrasaction='ignore')
        if nuovo:
            w.writeheader()
        v = dict(v)
        v['quando'] = time.strftime('%Y-%m-%d %H:%M:%S')
        w.writerow(v)
        f.flush()
        os.fsync(f.fileno())


def registra_impronta(run, riga):
    """Al primo fit riuscito di un run senza impronta attesa: la si registra e
    diventa l'attesa di ogni rilancio (il mondo deve restare lo stesso)."""
    if run['impronta_attesa'] or not riga or riga.get('esito') != 'ok':
        return
    imp = str(riga.get('impronta_input_data') or '')
    if not imp or imp == 'n/d':
        return
    nuovo = not (os.path.exists(IMPRONTE_DRIVE) and os.path.getsize(IMPRONTE_DRIVE) > 0)
    with open(IMPRONTE_DRIVE, 'a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['nome', 'codice', 'seed_mondo', 'impronta', 'quando'])
        if nuovo:
            w.writeheader()
        w.writerow({'nome': run['nome'], 'codice': run['codice'], 'seed_mondo': run['seed_mondo'],
                    'impronta': imp, 'quando': time.strftime('%Y-%m-%d %H:%M:%S')})


def verifica_sicura(run, riga, impronte):
    """Un errore nel CONTROLLO non deve fermare il ciclo: la riga del runner e'
    gia' su Drive; il controllo fallito viene registrato come esito
    'errore_verifica' con il messaggio, e si prosegue."""
    try:
        return verifica(run, riga, impronte)
    except Exception as e:
        import traceback
        traceback.print_exc()
        return {'n': run['n'], 'codice': run['codice'], 'nome': run['nome'], 'seed_mondo': run['seed_mondo'],
                'regola': run['regola'], 'atteso': '' if run['atteso'] is None else run['atteso'],
                'ottenuto': riga.get('rapporto_mediana') if riga else '', 'pit': riga.get('pit') if riga else '',
                'scostamento_pct': '', 'esito_scostamento': '', 'ammissibile': '', 'divergenze_pct': '',
                'problemi_config': '', 'esito': 'errore_verifica',
                'messaggio': 'il controllo della riga e' + A + ' fallito (' + type(e).__name__ + ': ' +
                             str(e)[:200] + '): la riga del runner e' + A + ' su Drive, va ricontrollata al ritorno'}


def segnala_anomalia(run, v):
    p = os.path.join(CARTELLA_DRIVE, 'ANOMALIA_' + run['nome'] + '.txt')
    testo = ('ANOMALIA sul run ' + str(run['n']) + ' (' + run['nome'] + ', disegno ' + run['codice'] + ')\n' +
             time.strftime('%Y-%m-%d %H:%M:%S') + '\n\nesito del controllo: ' + v['esito'] + '\n' +
             'atteso: ' + str(v['atteso']) + '   ottenuto: ' + str(v['ottenuto']) +
             '   scostamento: ' + str(v['scostamento_pct']) + '%\nmessaggio: ' + v['messaggio'] + '\n\n' +
             'Il ciclo e' + A + ' proseguito. La riga resta nel CSV del disegno con esito "anomalia": '
             'va guardata al ritorno.\nSe la causa e' + A + ' nella pipeline anche le altre righe sono sospette.\n')
    with open(p, 'w', encoding='utf-8') as f:
        f.write(testo)
    print()
    print('!' * 72)
    print('ANOMALIA - run ' + str(run['n']) + ' ' + run['nome'] + ' (il ciclo prosegue)')
    print(v['messaggio'])
    print('scritto ' + p)
    print('!' * 72)


# ============================================================================
# il ciclo
# ============================================================================
IMPRONTE = leggi_impronte()
fatti = {}
for _d in DISEGNI_ATTIVI:
    fatti.update(righe_ok(os.path.join(CARTELLA_DRIVE, INFO_DISEGNO[_d]['csv'])))
da_fare = [r for r in SEQUENZA if r['nome'] not in fatti]
print('cartella: ' + CARTELLA_DRIVE)
print('run gia' + A + ' fatti: ' + str(len([r for r in SEQUENZA if r['nome'] in fatti])) + ' su ' + str(len(SEQUENZA)))
print('run da fare (' + str(len(da_fare)) + '): ' + ', '.join(r['nome'] for r in da_fare))

t_inizio = time.time()
durate = []
for run in SEQUENZA:
    if run['nome'] in fatti:
        print('run ' + str(run['n']) + ' ' + run['nome'] + ': gia' + A + ' registrato con esito ok, salto.')
        continue
    rimasti = len(da_fare) - da_fare.index(run)
    print()
    print('=' * 72)
    print('RUN ' + str(run['n']) + ' - ' + run['nome'] + '   (' + str(SEQUENZA.index(run) + 1) + ' di ' +
          str(len(SEQUENZA)) + ')   disegno ' + run['codice'] + ' = ' + run['disegno'] +
          (' [' + run['parametri_disegno'] + ']' if run['parametri_disegno'] else ''))
    print('  mondo ' + str(run['seed_mondo']) + ' | prior ' + run['prior_tipo'] + ' sigma ' + str(run['prior_sigma']) +
          (' mu ' + str(run['prior_mu']) if run['prior_mu'] not in ('', None) else '') +
          ' | trattati: ' + (run['canali_trattati'] or '-') +
          (' | controllo baseline f=' + str(run['controllo_baseline'])
           if run.get('controllo_baseline') else '') +
          ' | dose: ' + ', '.join(k + ' ' + str(v) + '%' for k, v in run['dose_per_canale'].items()) +
          ' | regola ' + run['regola'] + (' (atteso ' + str(run['atteso']) + ')' if run['atteso'] is not None else ''))
    if durate:
        print('  trascorso: ' + durata(time.time() - t_inizio) + ' | stima per i ' + str(rimasti) +
              ' che restano: ' + durata(statistics.mean(durate) * rimasti))
    print('=' * 72)
    t_run = time.time()
    proc = subprocess.run(comando(run), stdin=subprocess.DEVNULL)
    dt = time.time() - t_run
    durate.append(dt)
    riga = riga_del_run(csv_di(run), run['nome'])
    v = verifica_sicura(run, riga, IMPRONTE)
    accoda_verifica(run, v)
    if riga is None:
        print('[ATTENZIONE] nessuna riga registrata (codice di uscita ' + str(proc.returncode) +
              '): processo morto. Proseguo: questo si rifa' + A + ' al rilancio.')
        continue
    if v['esito'] in ('ok', 'segnalato', 'non_ammissibile') and run['nome'] not in IMPRONTE:
        registra_impronta(run, riga)
        IMPRONTE = leggi_impronte()
    print('riga registrata: ' + run['nome'] + ' | esito=' + str(riga.get('esito')) +
          ' | mediana/vero=' + str(riga.get('rapporto_mediana')) + ' | pit=' + str(riga.get('pit')) +
          ' | nodi=' + str(riga.get('n_knots')) + ' | impronta=' + str(riga.get('impronta_input_data')) +
          ' | trattati=' + str(riga.get('canali_trattati')) + ' | div=' + str(v['divergenze_pct']) + '%')
    print('controllo: ' + v['esito'].upper() + ' - ' + v['messaggio'])
    print('tempo del run: ' + durata(dt) + ' | totale: ' + durata(time.time() - t_inizio))
    if v['esito'] == 'anomalia':
        segnala_anomalia(run, v)
    elif v['esito'] == 'errore_tecnico':
        print('[ATTENZIONE] errore tecnico: proseguo, questo viene ritentato al rilancio.')

print()
print('ciclo finito. Tempo totale: ' + durata(time.time() - t_inizio) +
      ('   (media per run: ' + durata(statistics.mean(durate)) + ' su ' + str(len(durate)) + ' fit fatti ora)'
       if durate else '   (nessun fit fatto ora: erano gia' + A + ' tutti registrati)'))

# ============================================================================
# RIEPILOGO per disegno + conteggi pre-registrati (meccanici, non letture)
# ============================================================================


def _med(valori):
    v = [x for x in valori if x is not None]
    return round(statistics.median(v), 4) if v else ''


IMPRONTE = leggi_impronte()
righe_riep = []
conteggi = []
per_disegno = {}
for _d in DISEGNI_ATTIVI:
    print()
    print('#' * 72)
    print('RIEPILOGO ' + _d + ' - ' + INFO_DISEGNO[_d]['titolo'])
    print('#' * 72)
    _csv = os.path.join(CARTELLA_DRIVE, INFO_DISEGNO[_d]['csv'])
    _fatti = righe_ok(_csv)
    _righe = [r for r in TABELLA if r['codice'] == _d and int(r['seed_mondo']) in MONDI_ATTIVI]
    print('  n  nome                          mondo   ottenuto     pit  scost.%  amm.   div.%  esito')
    _vs = []
    for run in _righe:
        riga = _fatti.get(run['nome']) or riga_del_run(_csv, run['nome'])
        v = verifica_sicura(run, riga, IMPRONTE)
        _vs.append((run, v, riga))
        righe_riep.append(v)
        seg = '' if v['esito'] == 'ok' else '  <-- ' + v['esito'].upper()
        print(' ' + str(v['n']).rjust(2) + '  ' + v['nome'].ljust(29) + ' ' + str(v['seed_mondo']).rjust(5) + ' ' +
              str(v['ottenuto']).rjust(9) + ' ' + (format(float(v['pit']), '.3f') if _numero(v['pit']) is not None else '-').rjust(7) +
              ' ' + (format(float(v['scostamento_pct']), '+.2f') if v['scostamento_pct'] not in ('', None) else '-').rjust(7) +
              '  ' + (v['ammissibile'] or '-').ljust(4) + ' ' +
              (str(v['divergenze_pct']) if v['divergenze_pct'] not in ('', None) else '-').rjust(6) + '  ' + v['esito'] + seg)
    # --- conteggi: mondi arrivati in fondo (ok / segnalato / non_ammissibile), poi solo ammissibili
    _arrivati = [(run, v, riga) for run, v, riga in _vs if v['esito'] in ('ok', 'segnalato', 'non_ammissibile')]
    _amm = [(run, v, riga) for run, v, riga in _arrivati if v['ammissibile'] == 'si']
    per_disegno[_d] = {}
    for _etichetta, _ins in (('tutti', _arrivati), ('ammissibili', _amm)):
        N = len(_ins)
        c_tot = sum(1 for run, v, riga in _ins if _numero(riga.get('pit')) is not None and 0.05 <= float(riga['pit']) <= 0.95)
        soglia = (6 if N == 8 else int(6.0 / 8.0 * N + 0.5)) if N else 0
        canali_tr = sorted({c for run, v, riga in _ins for c in (run['canali_trattati'].split(';') if run['canali_trattati'] else [])})
        c_k, larg_k = {}, {}
        for ch in canali_tr:
            dentro, larg = [], []
            for run, v, riga in _ins:
                for rc in righe_canali(run):
                    if rc['canale'] == ch and str(rc.get('trattato')) == '1':
                        dentro.append(str(rc.get('dentro')) == '1')
                        larg.append(_numero(rc.get('larghezza_relativa')))
            c_k[ch] = sum(dentro)
            larg_k[ch] = _med(larg)
        rapporti = [_numero(riga.get('rapporto_mediana')) for run, v, riga in _ins]
        rho = []
        for run, v, riga in _ins:
            rc = righe_canali(run)
            if rc:
                rho.append(_numero(rc[0].get('rho_spearman')))
        record = {'disegno': _d, 'insieme': _etichetta, 'N': N, 'soglia_identificazione': soglia, 'C_tot': c_tot,
                  'C_k': ';'.join(k + '=' + str(v) for k, v in c_k.items()),
                  'larghezza_mediana_k': ';'.join(k + '=' + str(v) for k, v in larg_k.items()),
                  'rapporto_mediano': _med(rapporti), 'rho_mediano': _med(rho),
                  'N_ok': sum(1 for k in c_k if c_k[k] >= soglia and soglia > 0)}
        conteggi.append(record)
        per_disegno[_d][_etichetta] = {'ins': _ins, 'c_k': c_k, 'larg_k': larg_k, 'N': N, 'soglia': soglia, 'c_tot': c_tot}
        print('  [' + _etichetta.ljust(11) + '] N=' + str(N) + ' soglia ' + str(soglia) + ' | C_tot=' + str(c_tot) +
              ' | C_k: ' + (', '.join(k + ' ' + str(v) for k, v in c_k.items()) or '-') +
              ' | larghezza mediana: ' + (', '.join(k + ' ' + str(v) for k, v in larg_k.items()) or '-') +
              ' | rapporto mediano ' + str(record['rapporto_mediano']) + ' | rho mediano ' + str(record['rho_mediano']))
    _n_ok = sum(1 for run, v, riga in _vs if v['esito'] == 'ok')
    _no = [v['nome'] + ' (' + v['esito'] + ')' for run, v, riga in _vs if v['esito'] not in ('ok', 'segnalato')]
    print('  ok ' + str(_n_ok) + ' su ' + str(len(_vs)) + (' | da guardare: ' + '; '.join(_no) if _no else ''))


# --- confronti fra disegni, mondo per mondo (solo se entrambi i disegni sono presenti)
def _larg_google(codice, seed):
    run = next((r for r in TABELLA if r['codice'] == codice and int(r['seed_mondo']) == seed and
                (codice != 'D3' or r['nome'].startswith('D3_geo_google_'))), None)
    if run is None:
        return None
    riga = riga_del_run(csv_di(run), run['nome'])
    if riga is None or riga.get('esito') != 'ok':
        return None
    for rc in righe_canali(run):
        if rc['canale'] == 'Google Ads':
            return _numero(rc.get('larghezza_relativa'))
    return None


print()
print('#' * 72)
print('CONFRONTI PRE-REGISTRATI FRA DISEGNI (Google Ads, stesso mondo)')
print('#' * 72)
for _nome_w, _a, _b, _cond in (('W7 (D7 <= D1: la dose non allarga)', 'D7', 'D1', 'le'),
                               ('W2 (D2 <= D7: trattare tutti insieme non costa)', 'D2', 'D7', 'le'),
                               ('W  (D1 geo < D4 pause2: il geografico identifica meglio)', 'D1', 'D4', 'lt')):
    if _a not in DISEGNI_ATTIVI or _b not in DISEGNI_ATTIVI:
        print(' ' + _nome_w + ': non calcolabile in questo lancio (' + _a + ' o ' + _b + ' non attivo)')
        continue
    coppie = []
    for seed in MONDI_ATTIVI:
        la, lb = _larg_google(_a, seed), _larg_google(_b, seed)
        if la is not None and lb is not None:
            coppie.append((seed, la, lb))
    w = sum(1 for s, la, lb in coppie if (la <= lb if _cond == 'le' else la < lb))
    print(' ' + _nome_w + ': ' + str(w) + ' su ' + str(len(coppie)) + ' mondi con entrambi i fit' +
          ('  [' + ', '.join(str(s) + ': ' + str(la) + ' vs ' + str(lb) for s, la, lb in coppie) + ']' if coppie else ''))
    conteggi.append({'disegno': _a + '_vs_' + _b, 'insieme': 'tutti', 'N': len(coppie), 'soglia_identificazione': 6,
                     'C_tot': '', 'C_k': _nome_w.split(' ')[0] + '=' + str(w), 'larghezza_mediana_k': '',
                     'rapporto_mediano': '', 'rho_mediano': '', 'N_ok': ''})

with open(RIEPILOGO, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=[c for c in COLONNE_VERIFICHE if c != 'quando'], extrasaction='ignore')
    w.writeheader()
    for v in righe_riep:
        w.writerow(v)
with open(CONTEGGI, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['disegno', 'insieme', 'N', 'soglia_identificazione', 'C_tot', 'C_k',
                                      'larghezza_mediana_k', 'rapporto_mediano', 'rho_mediano', 'N_ok'])
    w.writeheader()
    for c in conteggi:
        w.writerow(c)
n_ok = sum(1 for v in righe_riep if v['esito'] == 'ok')
n_no = [v['nome'] + ' (' + v['esito'] + ')' for v in righe_riep if v['esito'] not in ('ok', 'segnalato')]
n_amm = [v['nome'] for v in righe_riep if v['ammissibile'] == 'no']
print()
print('ok: ' + str(n_ok) + ' su ' + str(len(righe_riep)) + ' run dei disegni attivi' +
      (' | ANOMALIE, NON AMMISSIBILI o errori: ' + '; '.join(n_no) if n_no else ''))
print('criterio di ammissibilita' + A + ' (R-hat <= 1,1, ESS >= 100 su roi_m e beta_m): ' +
      ('tutti i run lo passano' if not n_amm else 'NON lo passano ' + str(len(n_amm)) + ': ' + ', '.join(n_amm) +
       ' (riportati, mai rifatti con campionatore diverso)'))
print()
print('#' * 72)
print('DA FARE ADESSO')
print('#' * 72)
print('1. Scarica da Drive la cartella ' + CARTELLA_DRIVE + ' per intera:')
print('   - griglia_<disegno>.csv, _canali.csv, _draw/, _verifiche.csv  (uno per disegno)')
print('   - griglia_riepilogo.csv, griglia_conteggi.csv, impronte_griglia.csv, attesi_griglia.csv')
print('   - preregistrazioni/  (le pre-registrazioni come scritte prima del primo fit)')
print('2. Riporta in chat i riepiloghi qui sopra per intero. I rami si leggono SOLO sulle')
print('   pre-registrazioni approvate, dopo il commit dei risultati.')
